# Paper-scale Breakout DQN — strict assignment baseline

Same recipe as the successful Pong run, applied to **Breakout** (assignment §4.3 additional environment). Target: paper DQN average **168**, best episode **225** (10M frames).

## Prerequisites
1. Commit and push the current working tree (`git push origin MyBranch`).
2. Kaggle: **GPU (T4 x2)**, **Internet = On**.
3. Estimated wall clock: **8–11 h** (Breakout episodes are longer than Pong; evaluation every 250k decisions with 5 games limits overhead). If your session limit is shorter, lower `total_epoch` to e.g. `2000000` and note the reduced budget.


In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
import os
if not os.path.isdir('/kaggle/working/DQN'):
    !git clone https://github.com/Sokach112/DQN.git
%cd /kaggle/working/DQN
!git checkout MyBranch || echo "WARNING: MyBranch not found; using the default branch"
!git log --oneline -3
!pip install -q -r requirements.txt


In [ ]:
from omegaconf import OmegaConf
cfg = OmegaConf.load('config/config.yaml').algorithm
defaults = {k: cfg.get(k) for k in ('architecture', 'activation', 'loss', 'use_target', 'double_dqn')}
print(defaults)
assert defaults == {'architecture': 'paper2013', 'activation': 'relu', 'loss': 'mse', 'use_target': True, 'double_dqn': False}, \
    "The clone is not the assignment-baseline code: commit and push the current working tree, then rerun this cell."
print('assignment-baseline defaults confirmed')


## Short Breakout smoke test (pipeline + ROM check)

20k decisions only; confirms the environment/wrappers work before spending GPU hours.

In [ ]:
!python main.py total_epoch=20000 interact_per_epoch=1 \
  env.name=BreakoutNoFrameskip-v4 env.num_testing_envs=1 \
  algorithm.buffer_size=20000 algorithm.learning_starts=20000 \
  algorithm.batch_size=32 algorithm.epsilon_timestep=250000 \
  test_interval=0 save_interval=0 use_tensorboard=false \
  +save_pth=/kaggle/working/dqn_checkpoints/smoke_breakout.pth 2>&1 | tail -n 20


## Main run — 2,500,000 decisions (10M frames), 2,500,000 gradient updates

Evaluation every 250k decisions (5 complete games) to keep long-episode evaluation overhead bounded.

In [ ]:
!python main.py \
  total_epoch=2500000 interact_per_epoch=1 \
  env.name=BreakoutNoFrameskip-v4 env.num_testing_envs=1 \
  algorithm.architecture=paper2013 algorithm.activation=relu \
  algorithm.loss=mse algorithm.use_target=true algorithm.double_dqn=false \
  algorithm.buffer_size=200000 algorithm.learning_starts=20000 \
  algorithm.learning_rate=1e-4 algorithm.batch_size=32 \
  algorithm.epsilon_timestep=250000 algorithm.end_epsilon=0.1 \
  algorithm.target_update_interval=1000 \
  test_interval=250000 test_episodes=5 save_interval=250000 \
  use_tensorboard=false \
  +save_pth=/kaggle/working/dqn_checkpoints/breakout_paper2013.pth \
  2>&1 | tee /kaggle/working/breakout_paper2013_training.log


## Independent verification (20 complete games on fresh seeds)

Breakout games are long, so 20 games is a compromise; increase `--games 40` if time allows.

In [ ]:
!python kaggle_results_analysis.py \
  --log /kaggle/working/breakout_paper2013_training.log \
  --checkpoint /kaggle/working/dqn_checkpoints/breakout_paper2013.pth \
  --output /kaggle/working/breakout_paper2013_analysis \
  --game BreakoutNoFrameskip-v4 --games 20 --device cuda \
  --reference-lines 168 225 --reference-labels "paper average (168)" "paper best episode (225)"


In [ ]:
!mkdir -p /kaggle/working/artifacts
!cp /kaggle/working/breakout_paper2013_training.log /kaggle/working/artifacts/
!cp /kaggle/working/dqn_checkpoints/breakout_paper2013.pth /kaggle/working/artifacts/
!cp -r /kaggle/working/breakout_paper2013_analysis /kaggle/working/artifacts/
!tar -czf /kaggle/working/breakout_paper2013_artifacts.tgz -C /kaggle/working artifacts
!ls -lh /kaggle/working/breakout_paper2013_artifacts.tgz


## Interpretation guide

- The paper's 168 is the **average** over evaluation games; 225 is its best episode. Breakout scores are much noisier than Pong.
- If the strict `paper2013` network underperforms, report the measured curve honestly; the earlier successful Pong run used the `nature2015` network, and architecture is the known difference (see the Kaggle Pong report).
- Download `breakout_paper2013_artifacts.tgz` and hand it back for independent verification and integration.